# MC Sim — GPU Build & Validation

Build and validate the molecular communication GPU simulator.
Progresses from basic correctness through performance comparison to large-scale stress tests.

**Before running:**
1. Runtime > Change runtime type > GPU (T4 or A100)
2. Click the 🔑 key icon, add a secret named `GITHUB_PAT` with your token, toggle notebook access on

In [ ]:
# 0. Setup: verify GPU, clone, build
!nvidia-smi | head -4
!nvcc --version | tail -1

from google.colab import userdata
PAT = userdata.get('GITHUB_PAT')
!git clone https://{PAT}@github.com/alwaysEpic/molecular_modeling_gpu.git
%cd molecular_modeling_gpu

!mkdir -p build && cd build && cmake .. 2>&1 | tail -2 && make -j$(nproc) 2>&1 | tail -3
!g++ -O2 -o build/dump_rng_cpu scripts/dump_rng_cpu.cpp -lm
!pip install -q numpy matplotlib scipy
print()
!ls -la build/mc_sim build/mc_sim_cpu

### Single Particle Path (thesis Figure 4.3)

Visualize the 3D random walk of a single molecule under Brownian motion.

In [ ]:
# Generate single particle path (everything mode, 1 particle, no drift)
!cd build && ./mc_sim -i 1 -e -n -t 1E-3 -W > /dev/null 2>&1

import numpy as np
import matplotlib.pyplot as plt

data = []
with open('build/output_gpu.csv') as f:
    for line in f:
        parts = line.strip().rstrip(',').split(',')
        if len(parts) >= 3:
            data.append([float(x) for x in parts[:3]])
pos = np.array(data)
pos_um = pos * 1e6  # convert to micrometers

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(pos_um[:, 0], pos_um[:, 1], pos_um[:, 2], linewidth=0.5, alpha=0.8)
ax.scatter(*pos_um[0], color='green', s=40, zorder=5, label='Start')
ax.scatter(*pos_um[-1], color='red', s=40, zorder=5, label='End')
ax.set_xlabel('x (μm)', fontsize=11)
ax.set_ylabel('y (μm)', fontsize=11)
ax.set_zlabel('z (μm)', fontsize=11)
ax.set_title('Single Particle Path — Brownian Motion (no drift)', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('particle_path.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Path: {len(pos)} steps, displacement: {np.linalg.norm(pos[-1]-pos[0])*1e6:.3f} μm")

---
## 1. Analytical Validation

Compare simulation output against known analytical solutions.
These are the same tests used in the thesis (Figures 4.1, 4.2, 4.7) but with
rigorous KS statistical testing instead of visual inspection.

### 1a. 1D First-Hit with Drift (thesis Figure 4.2)

Particles diffuse + drift in 1D toward a planar receiver at distance b=300nm.
First-passage time follows an inverse Gaussian distribution (thesis eq 4.3).

In [ ]:
# 10k paths, 100k timesteps — long kernel
!cd build && ./mc_sim -i 10000 -f -l 3E-7 -t 1E-2 -v
print()
!python scripts/validate_1d_firsthit.py build/output_gpu.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2
from IPython.display import Image, display
display(Image('validation_1d_firsthit.png'))

### 1b. 3D Spherical Receiver, No Drift (thesis Figure 4.1)

Particles diffuse in 3D toward a spherical absorber (a=10nm at d=50nm).
Hit probability is a/d = 0.2, conditional hit-time follows Schulten eq 3.108.

In [ ]:
# 10k paths, 10k timesteps — long kernel
!cd build && ./mc_sim -i 10000 -f -n -v
print()
!python scripts/validate_3d_diffusion.py build/output_gpu.csv \
    --total-paths 10000
from IPython.display import Image, display
display(Image('validation_3d_diffusion.png'))

### 1c. Wall Reflection (thesis Figure 4.7)

Transmitter and receiver positioned 200nm from the vessel wall (r=8μm).
Validates that cylindrical wall reflection does not corrupt diffusion statistics.

The thesis showed a visible discrepancy here (boundary-crossing bias).
The Brownian bridge correction fixes this.

In [ ]:
# 10k paths, wall reflection, no drift
!cd build && ./mc_sim -i 10000 -f -w -n \
    --start-y 7.8E-6 \
    --rec-y 7.8E-6 --rec-z 50E-9 \
    -r 8E-6 -t 0.4E-3 -v
print()
!python scripts/validate_3d_walls.py build/output_gpu.csv \
    --total-paths 10000
from IPython.display import Image, display
display(Image('validation_3d_walls.png'))

---
## 2. Consistency Checks

Verify that CPU and GPU paths agree, both kernel architectures produce
the same distribution, and the RNG is statistically sound.

In [ ]:
# CPU vs GPU Long kernel agreement
print("=== CPU vs GPU Long ===")
!cd build && ./mc_sim -i 5000 -c -f -l 3E-7 -t 1E-2 > /dev/null 2>&1
!python scripts/validate_cpu_gpu_agreement.py build/output_h.csv build/output_gpu.csv \
    --timestep 1E-7 --no-plot
print()
# CPU vs GPU Wide kernel agreement
print("=== CPU vs GPU Wide ===")
!cd build && ./mc_sim -i 5000 -c -f -l 3E-7 -t 1E-2 -W > /dev/null 2>&1
!python scripts/validate_cpu_gpu_agreement.py build/output_h.csv build/output_gpu.csv \
    --timestep 1E-7 --no-plot

In [ ]:
# GPU reproducibility (same seed = identical output)
!cd build && ./mc_sim -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_gpu.csv run1.csv
!cd build && ./mc_sim -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_gpu.csv run2.csv
!diff build/run1.csv build/run2.csv && echo 'REPRODUCIBILITY: PASS' || echo 'REPRODUCIBILITY: FAIL'

In [ ]:
# RNG quality — GPU (Philox) and CPU (Box-Muller)
!cd build && ./dump_rng_cpu 10000 > rng_cpu.csv
print("=== CPU RNG ===")
!python scripts/validate_rng.py build/rng_cpu.csv --no-plot

In [ ]:
%%writefile /tmp/dump_rng_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <curand.h>
#include <curand_kernel.h>

__global__ void gen_randn(curandStatePhilox4_32_10_t* rng, float* out, int n, int draws_per_thread) {
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx >= n) return;
  for (int d = 0; d < draws_per_thread; d++) {
    out[idx * draws_per_thread + d] = curand_normal(&rng[idx]);
  }
}

__global__ void init_rng(curandStatePhilox4_32_10_t* rng, long long seed, int n) {
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx >= n) return;
  curand_init(seed, idx, 0, &rng[idx]);
}

int main(int argc, char** argv) {
  int n_threads = 1000;
  int draws = 10;
  int total = n_threads * draws;
  long long seed = 42;
  if (argc > 1) seed = atoll(argv[1]);

  curandStatePhilox4_32_10_t* d_rng;
  float* d_out;
  cudaMalloc(&d_rng, n_threads * sizeof(curandStatePhilox4_32_10_t));
  cudaMalloc(&d_out, total * sizeof(float));

  int block = 128;
  int grid = (n_threads + block - 1) / block;
  init_rng<<<grid, block>>>(d_rng, seed, n_threads);
  cudaDeviceSynchronize();
  gen_randn<<<grid, block>>>(d_rng, d_out, n_threads, draws);
  cudaDeviceSynchronize();

  float* out = (float*)malloc(total * sizeof(float));
  cudaMemcpy(out, d_out, total * sizeof(float), cudaMemcpyDeviceToHost);

  for (int i = 0; i < total; i++) printf("%0.15f\n", out[i]);

  free(out);
  cudaFree(d_rng);
  cudaFree(d_out);
  return 0;
}

In [ ]:
!nvcc -O2 -o build/dump_rng_gpu /tmp/dump_rng_gpu.cu
!cd build && ./dump_rng_gpu 42 > rng_gpu.csv
print("=== GPU RNG ===")
!python scripts/validate_rng.py build/rng_gpu.csv --no-plot

---
## 3. Performance Comparison

Compare execution time across the original thesis code, the optimized wide
kernel, and the long kernel. The thesis-baseline branch contains the
unmodified GPU code running on the same hardware.

The thesis had two kernel architectures:
- **Wide**: one kernel launch per timestep, all particles advance together
- **Long**: single launch, each thread runs one particle's full path

In [ ]:
# Build thesis-baseline for comparison
import os
from google.colab import userdata

if not os.path.isdir('/content/baseline_build'):
    PAT = userdata.get('GITHUB_PAT')
    !git clone https://{PAT}@github.com/alwaysEpic/molecular_modeling_gpu.git /content/baseline_build
    !cd /content/baseline_build && git checkout thesis-baseline
    !cd /content/baseline_build && mkdir -p build && cd build && cmake .. 2>&1 | tail -1 && make -j$(nproc) 2>&1 | tail -1
else:
    print("Baseline already built")

In [ ]:
import subprocess, re, os

def run_gpu_time(binary_path, args, cwd, runs=3):
    """Run binary multiple times, extract GPU time, return median."""
    times = []
    for _ in range(runs):
        result = subprocess.run(
            [binary_path] + args,
            capture_output=True, text=True, cwd=cwd
        )
        output = result.stdout + result.stderr
        match = re.search(r'GPU code execution time is ([\d.]+)s', output)
        if match:
            times.append(float(match.group(1)))
    return sorted(times)[len(times)//2] if times else None

current_dir = os.getcwd()
current_bin = os.path.join(current_dir, "build", "mc_sim")
baseline_bin = "/content/baseline_build/build/mc_sim"

test_args_1d = ["-f", "-l", "3E-7", "-t", "1E-2", "-v"]
particle_counts = [1000, 5000, 10000]

# Thesis GTX 1070 numbers from Table 4.2 (with drift, first-hit)
thesis_1070 = {1000: 2.27, 5000: 3.46, 10000: 4.81}

print("=" * 90)
print("Thesis vs Current: 1D First-Hit with Drift (b=300nm, 100k steps)")
print("Same GPU, median of 3 runs")
print("=" * 90)
print()
print(f"{'Particles':<12} {'Thesis 1070':>12} {'Thesis GPU':>12} {'Wide GPU':>12} {'Long GPU':>12} {'Code Speedup':>14}")
print(f"{'':.<12} {'(Table 4.2)':>12} {'(baseline)':>12} {'(current)':>12} {'(current)':>12} {'(wide/base)':>14}")
print("-" * 90)

for n in particle_counts:
    args = ["-i", str(n)] + test_args_1d
    t_base = run_gpu_time(baseline_bin, args, "/content/baseline_build/build")
    t_wide = run_gpu_time(current_bin, args + ["-W"], os.path.join(current_dir, "build"))
    t_long = run_gpu_time(current_bin, args, os.path.join(current_dir, "build"))
    t_1070 = thesis_1070.get(n, None)
    code_speedup = f"{t_base/t_wide:.1f}x" if (t_base and t_wide) else "—"
    print(f"{n:<12,} {t_1070 or 0:>11.3f}s {t_base or 0:>11.3f}s {t_wide or 0:>11.3f}s {t_long or 0:>11.3f}s {code_speedup:>14}")

print()
print("Code Speedup = thesis-baseline / current wide (same GPU)")

### Performance Charts (thesis Figures 4.4 / 4.5)

Execution time and speedup multiplier vs particle count.

In [ ]:
import subprocess, re, os
import matplotlib.pyplot as plt
import numpy as np

def run_gpu_time_chart(binary_path, args, cwd, runs=3):
    times = []
    for _ in range(runs):
        result = subprocess.run([binary_path] + args, capture_output=True, text=True, cwd=cwd)
        match = re.search(r'GPU code execution time is ([\d.]+)s', result.stdout + result.stderr)
        if match:
            times.append(float(match.group(1)))
    return sorted(times)[len(times)//2] if times else None

def run_cpu_time_chart(binary_path, args, cwd, runs=2):
    times = []
    for _ in range(runs):
        result = subprocess.run([binary_path] + args, capture_output=True, text=True, cwd=cwd)
        match = re.search(r'CPU code execution time.*?is ([\d.]+)', result.stdout + result.stderr)
        if match:
            times.append(float(match.group(1)))
    return sorted(times)[len(times)//2] if times else None

current_dir = os.getcwd()
current_gpu = os.path.join(current_dir, "build", "mc_sim")
current_cpu = os.path.join(current_dir, "build", "mc_sim_cpu")
baseline_gpu = "/content/baseline_build/build/mc_sim"

test_args = ["-f", "-l", "3E-7", "-t", "1E-2", "-v"]
particle_counts = [100, 500, 1000, 2000, 5000, 10000]

cpu_times, thesis_gpu_times, wide_times, long_times = [], [], [], []

print("Collecting timing data...")
for n in particle_counts:
    args = ["-i", str(n)] + test_args
    print(f"  {n} particles...", end=" ", flush=True)
    t_cpu = run_cpu_time_chart(current_cpu, args, os.path.join(current_dir, "build"))
    t_thesis = run_gpu_time_chart(baseline_gpu, args, "/content/baseline_build/build")
    t_wide = run_gpu_time_chart(current_gpu, args + ["-W"], os.path.join(current_dir, "build"))
    t_long = run_gpu_time_chart(current_gpu, args, os.path.join(current_dir, "build"))
    cpu_times.append(t_cpu)
    thesis_gpu_times.append(t_thesis)
    wide_times.append(t_wide)
    long_times.append(t_long)
    print(f"CPU={t_cpu:.3f}s  Thesis={t_thesis:.3f}s  Wide={t_wide:.3f}s  Long={t_long:.3f}s")

N = np.array(particle_counts)
cpu = np.array(cpu_times)
thesis = np.array(thesis_gpu_times)
wide = np.array(wide_times)
long_arr = np.array(long_times)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(N, cpu, 'b-o', linewidth=2, markersize=6, label='CPU')
ax1.plot(N, thesis, 'r--s', linewidth=2, markersize=6, label='Thesis GPU (wide)')
ax1.plot(N, wide, 'g-^', linewidth=2, markersize=6, label='Wide kernel')
ax1.plot(N, long_arr, 'k-d', linewidth=2, markersize=6, label='Long kernel')
ax1.set_xlabel('Number of Paths', fontsize=12)
ax1.set_ylabel('Time (s)', fontsize=12)
ax1.set_title('Execution Time vs Particle Count\n(1D first-hit, 100k steps)', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

speedup_thesis = cpu / thesis
speedup_wide = cpu / wide
speedup_long = cpu / long_arr

ax2.plot(N, speedup_thesis, 'r--s', linewidth=2, markersize=6, label='Thesis GPU vs CPU')
ax2.plot(N, speedup_wide, 'g-^', linewidth=2, markersize=6, label='Wide kernel vs CPU')
ax2.plot(N, speedup_long, 'k-d', linewidth=2, markersize=6, label='Long kernel vs CPU')
ax2.axhline(y=1, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Number of Paths', fontsize=12)
ax2.set_ylabel('Speedup (CPU time / GPU time)', fontsize=12)
ax2.set_title('GPU Speedup vs Particle Count\n(1D first-hit, 100k steps)', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.savefig('performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPeak speedup: Long={max(speedup_long):.0f}x  Wide={max(speedup_wide):.0f}x  Thesis={max(speedup_thesis):.1f}x")

### Scale Sweep (thesis Table 4.3)

Long kernel execution time from 10k to 5M paths.
The thesis swept 5M-500M on a GTX 1070 (5M took 5.08s).

In [ ]:
import subprocess, re, os
import matplotlib.pyplot as plt
import numpy as np

def run_gpu_sweep(binary_path, args, cwd):
    result = subprocess.run([binary_path] + args, capture_output=True, text=True, cwd=cwd)
    match = re.search(r'GPU code execution time is ([\d.]+)s', result.stdout + result.stderr)
    return float(match.group(1)) if match else None

current_dir = os.getcwd()
current_bin = os.path.join(current_dir, "build", "mc_sim")

# Default first-hit, no drift (matches thesis Table 4.3: 10k steps, 1E-3 end time)
test_args = ["-f", "-n", "-v"]
sweep_counts = [10000, 50000, 100000, 500000, 1000000, 2500000, 5000000]

# Thesis GTX 1070 long kernel times from Table 4.3
thesis_long_1070 = {5000000: 5.08}

print("=" * 70)
print("Scale Sweep: Long Kernel (default first-hit, no drift, 10k steps)")
print("=" * 70)
print()
print(f"{'Paths':<12} {'Long GPU':>12} {'Thesis 1070':>14}")
print("-" * 40)

long_times = []
for n in sweep_counts:
    args = ["-i", str(n)] + test_args
    t = run_gpu_sweep(current_bin, args, os.path.join(current_dir, "build"))
    long_times.append(t)
    t_thesis = thesis_long_1070.get(n, None)
    thesis_str = f"{t_thesis:.2f}s" if t_thesis else "—"
    print(f"{n:<12,} {t:>11.3f}s {thesis_str:>14}")

fig, ax = plt.subplots(figsize=(8, 5))
N = np.array(sweep_counts)
T = np.array(long_times)

ax.plot(N, T, 'k-d', linewidth=2, markersize=6, label='Long kernel')
ax.set_xlabel('Number of Paths', fontsize=12)
ax.set_ylabel('Time (s)', fontsize=12)
ax.set_title('Long Kernel Scaling (default first-hit, 10k steps)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('scale_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nThroughput at 5M: {5e6/T[-1]:.0f} paths/sec")

---
## 4. Stress Tests

High-sample-count validation with maximum statistical power.
At 1M paths the KS test detects CDF discrepancies above 0.2%.
At 10M paths (50x beyond thesis maximum of 200k) the KS critical value
is ~0.0006.

In [ ]:
# 1M paths — 1D first-hit with drift
!cd build && ./mc_sim -i 1000000 -f -l 3E-7 -t 1E-2 -v
print()
!python scripts/validate_1d_firsthit.py build/output_gpu.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2
from IPython.display import Image, display
display(Image('validation_1d_firsthit.png'))

In [ ]:
# 1M paths — 3D spherical receiver
!cd build && ./mc_sim -i 1000000 -f -n -v
print()
!python scripts/validate_3d_diffusion.py build/output_gpu.csv \
    --total-paths 1000000
from IPython.display import Image, display
display(Image('validation_3d_diffusion.png'))

In [ ]:
# 100k paths — wall reflection stress test
!cd build && ./mc_sim -i 100000 -f -w -n \
    --start-y 7.8E-6 \
    --rec-y 7.8E-6 --rec-z 50E-9 \
    -r 8E-6 -t 0.4E-3 -v
print()
!python scripts/validate_3d_walls.py build/output_gpu.csv \
    --total-paths 100000
from IPython.display import Image, display
display(Image('validation_3d_walls.png'))

### 10M Paths — Maximum Scale

The thesis maximum was 200k paths (350s on a GTX 1070).
With the long kernel we can simulate 50x more.

In [ ]:
# 10M paths — 1D first-hit
import time
print("Starting 10M path simulation...")
t0 = time.time()
!cd build && ./mc_sim -i 10000000 -f -l 3E-7 -t 1E-2 -v
t1 = time.time()
print(f"\nTotal wall time: {t1-t0:.1f}s")
print()
!python scripts/validate_1d_firsthit.py build/output_gpu.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2
from IPython.display import Image, display
display(Image('validation_1d_firsthit.png'))